# Prometheus - Medical Image Segmentation Training on PUMA

**Dataset:** [PUMA](https://puma.grand-challenge.org/dataset/) — Panoptic segmentation of nUclei and tissue in MelanomA

**Models:** UNetTissue (6 tissue classes) | DualUNet (6 tissue + 10 nuclei classes)

**GPU:** Google Colab G4 / A100 100GB VRAM

---
## 1. Setup

In [ ]:
import os, sys

import torch

print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}")
for i in range(torch.cuda.device_count()):
    print(f"  [{i}] {torch.cuda.get_device_name(i)} "
          f"({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
REPO_DIR = "/content/prometheus"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hoangtung386/Prometheus.git {REPO_DIR}

%cd {REPO_DIR}
!pip install -e .
!pip install tensorboard matplotlib
sys.path.insert(0, REPO_DIR)

In [ ]:
from prometheus import (
    UNetTissue,
    DualUNet,
    CombinedLoss,
    Trainer,
    visualize_sample,
    show_prediction,
    predict_sample,
)
from prometheus.config import ModelConfig, TrainingConfig, DEFAULT_CONFIG, DEFAULT_TRAINING_CONFIG
from prometheus.data import (
    PUMADataset,
    create_puma_dataloaders,
    train_transform,
    val_transform,
    collate_puma,
)
from prometheus.data.puma_dataset import (
    geojson_to_mask,
    TISSUE_CLASSES,
    NUCLEI_CLASSES,
    TISSUE_CLASS_TO_IDX,
    NUCLEI_CLASS_TO_IDX,
)
from prometheus.data.transforms import (
    Compose,
    RandomHorizontalFlip,
    RandomVerticalFlip,
    RandomRotate90,
    ElasticDeformation,
    NormalizeTile,
    RandomBrightnessContrast,
    RandomGaussianNoise,
    Normalize,
)
from prometheus.models import (
    Encoder,
    Decoder,
    TissueAttentionEncoder,
    TissueDecoder,
    build_encoder,
    forward_encoder,
    build_decoder,
    forward_decoder,
)
from prometheus.training import (
    dice_score,
    warmup_cosine_lr,
    train_one_epoch,
    validate,
)
from prometheus.losses import (
    BCEWithLogitsLoss,
    DiceLoss,
    FocalLoss,
    MultiClassDiceLoss,
    TverskyLoss,
)
from prometheus.blocks import (
    ConvNeXtBlock,
    DecoderBlock,
    LocalGlobalAttention,
    SparseMoE,
    Expert,
    EncoderTransformerBlock,
    EncoderTransformerStack,
)
from prometheus.utils import LayerNorm, GRN

print("All imports from prometheus codebase successful!")

---
## 2. Configuration

In [ ]:
model_cfg = ModelConfig(
    in_chans=3,
    num_classes=11,
    drop_path_rate=0.1,
)

train_cfg = TrainingConfig(
    model_type="DualUNet",
    image_size=1024,
    num_tissue_classes=6,
    num_nuclei_classes=10,
    batch_size=16,
    epochs=200,
    lr=2e-4,
    weight_decay=1e-2,
    warmup_epochs=10,
    gradient_clip_norm=1.0,
    amp=True,
    data_root="/content/drive/MyDrive/puma_data",
    num_workers=8,
    val_split=0.1,
    log_dir="/content/drive/MyDrive/prometheus_logs",
    ckpt_dir="/content/drive/MyDrive/prometheus_checkpoints",
    log_interval=10,
    save_interval=5,
    moe_loss_weight=0.01,
)

os.makedirs(train_cfg.log_dir, exist_ok=True)
os.makedirs(train_cfg.ckpt_dir, exist_ok=True)

print(f"Model: {train_cfg.model_type}")
print(f"Tissue classes: {train_cfg.num_tissue_classes}, Nuclei classes: {train_cfg.num_nuclei_classes}")
print(f"Image size: {train_cfg.image_size}x{train_cfg.image_size}, Batch: {train_cfg.batch_size}")

---
## 3. Dataset — PUMA (tissue 6 classes + nuclei 10 classes)

Dataset yêu cầu:
```
data_root/
├── images/              # 206 *.tif (1024x1024, 40x H&E)
├── geojson_tissue/      # 206 *_tissue.geojson (6 classes)
└── geojson_nuclei/      # 206 *_nuclei.geojson (10 classes)
```

**Tissue classes:** background(0), tumor(1), stroma(2), epidermis(3), necrosis(4), blood_vessel(5)

**Nuclei classes:** background(0), tumor(1), stroma(2), vascular_endothelium(3), histiocyte(4), melanophage(5), lymphocyte(6), plasma_cell(7), neutrophil(8), apoptotic_cell(9), epithelium(10)

In [ ]:
if os.path.exists(train_cfg.data_root):
    train_loader, val_loader = create_puma_dataloaders(
        root=train_cfg.data_root,
        image_size=train_cfg.image_size,
        batch_size=train_cfg.batch_size,
        num_workers=train_cfg.num_workers,
        val_split=train_cfg.val_split,
    )
else:
    print(f"Data not found at {train_cfg.data_root}")
    print("Falling back to dummy data for testing...")
    train_loader = val_loader = None

### 3a. Quick visualization

In [ ]:
if os.path.exists(train_cfg.data_root):
    ds = PUMADataset(root=train_cfg.data_root, image_size=train_cfg.image_size)
    visualize_sample(ds, 0)

### 3b. In-memory mask caching
Để tránh render GeoJSON polygon → mask mỗi epoch, dataset đã cache mask trong RAM lần đầu.
Với 206 ROI × (6 + 11) channels × 1024² ≈ 3.6 GB — cần heap RAM > 12 GB (Colab G4 OK).
Nếu OOM → tắt cache: `PUMADataset(..., cache_masks=False)`

---
## 4. Model

### Lưu ý về số classes:
- **UNetTissue** nhận `num_classes` duy nhất → chỉ train tissue với 6 classes
- **DualUNet** hiện tại dùng chung `num_classes` cho cả 2 head → cần custom model cho 6 & 10
  → Tạm thời train từng head riêng, hoặc modify model sau.

In [ ]:
if train_cfg.model_type == "UNetTissue":
    model_cfg = ModelConfig(
        in_chans=3,
        num_classes=train_cfg.num_tissue_classes,
        drop_path_rate=0.1,
    )
    model = UNetTissue(config=model_cfg)
else:
    model_cfg = ModelConfig(
        in_chans=3,
        num_classes=train_cfg.num_nuclei_classes + 1,
        drop_path_rate=0.1,
    )
    model = DualUNet(config=model_cfg)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{train_cfg.model_type}: {total:,} params ({trainable:,} trainable)")

---
## 5. Training với Trainer

In [ ]:
if train_loader is None:
    print("=== DUMMY DATA MODE ===")
    from torch.utils.data import DataLoader
    dummy = [(torch.randn(3, 256, 256),
              {"tissue": torch.randint(0, 2, (6, 256, 256)).float(),
               "nuclei": torch.randint(0, 2, (11, 256, 256)).float()})
             for _ in range(64)]
    train_loader = val_loader = DataLoader(dummy, batch_size=4, shuffle=True)
    train_cfg.epochs = 5

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_cfg,
    device=device,
)

best_dice = trainer.fit()
print(f"Training completed! Best dice: {best_dice:.4f}")

---
## 6. Inference & Visualization

In [ ]:
if os.path.exists(train_cfg.data_root):
    ds = PUMADataset(root=train_cfg.data_root, image_size=train_cfg.image_size, augment=False)
    show_prediction(model, ds, 0, train_cfg.model_type, device)

---
## 7. Data Preprocessing Utilities

Các utility functions và classes đã được import từ codebase:

In [ ]:
print("Available utilities from prometheus codebase:")
print(f"  Tissue classes ({len(TISSUE_CLASSES)}): {TISSUE_CLASSES}")
print(f"  Nuclei classes ({len(NUCLEI_CLASSES)}): {NUCLEI_CLASSES}")
print()
print("Transforms:")
print(f"  Compose, RandomHorizontalFlip, RandomVerticalFlip, RandomRotate90")
print(f"  ElasticDeformation, NormalizeTile, RandomBrightnessContrast, RandomGaussianNoise, Normalize")
print(f"  train_transform: {train_transform()}")
print(f"  val_transform: {val_transform()}")
print()
print("Models:")
print(f"  UNetTissue, DualUNet, Encoder, Decoder, TissueAttentionEncoder, TissueDecoder")
print(f"  build_encoder, forward_encoder, build_decoder, forward_decoder")
print()
print("Loss functions:")
print(f"  CombinedLoss, BCEWithLogitsLoss, DiceLoss, FocalLoss, MultiClassDiceLoss, TverskyLoss")
print()
print("Building blocks:")
print(f"  ConvNeXtBlock, DecoderBlock, LocalGlobalAttention, SparseMoE, Expert")
print(f"  EncoderTransformerBlock, EncoderTransformerStack")
print()
print("Training:")
print(f"  Trainer, dice_score, warmup_cosine_lr, train_one_epoch, validate")
print()
print("Visualization:")
print(f"  visualize_sample, predict_sample, show_prediction")
print()
print("Utilities:")
print(f"  LayerNorm, GRN")
print()
print("Config defaults:")
print(f"  DEFAULT_CONFIG: {DEFAULT_CONFIG}")
print(f"  DEFAULT_TRAINING_CONFIG: {DEFAULT_TRAINING_CONFIG}")

---
## 8. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {train_cfg.log_dir}